# LLM Fact-Checking on LIAR / 用大语言模型核查 LIAR 陈述

`07` 表明：去掉来源捷径后，只看文字的模型（TF-IDF、DistilBERT）在 LIAR 上只有 AUC 约 0.68。一句话是真是假，往往取决于文字以外的事实。这一步检验两件事：

| 实验 | 给模型看什么 | 要回答的问题 |
|---|---|---|
| **5a** | 只有陈述 | 大模型自带的世界知识，能否超过只看文字的模型？ |
| **5b** | 陈述 + 为它检索到的 3 段维基百科 | 外部证据能否突破 0.68 的上限？ |
| **对照组** | 陈述 + **另一条**陈述的 3 段维基百科 | 5b 的变化来自证据的 *内容*，还是只来自"提示变长、格式变了"？ |

模型是 Claude Haiku 4.5（最便宜的型号），通过 Batch API 提交（五折），temperature 0，结构化输出（只能返回 `{"probability": 整数}`）。三个实验只在 LIAR test 的 1,283 条陈述上运行，**都没有在测试集上调过任何东西**。代码：`src/llm_factcheck.py`、`src/evidence.py`。

`07` showed text-only models reach only AUC ~0.68 on LIAR. Here Claude Haiku 4.5 judges the same 1,283 test statements (5a) alone, (5b) with three retrieved Wikipedia passages, and (control) with another statement's passages. Batch API, temperature 0, structured output; nothing was tuned on test.

## 1. Setup / 环境

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path.cwd().parent))

from src import llm_factcheck as L
from src.analysis import paired_bootstrap_auc_diff
from src.baselines import bootstrap_ci, evaluate
from src.evidence import load_evidence
from src.external import load_liar

LLM, LIAR_RES = Path('../results/llm'), Path('../results/liar')
ORDER = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']
pd.set_option('display.width', 200, 'display.max_colwidth', 120)

liar = load_liar().set_index('doc_id')

def probs(path, key='LIAR test'):
    d = np.load(path, allow_pickle=True)
    return pd.Series(d[f'prob::{key}'], index=d[f'doc_id::{key}'].astype(str))

def replies(path, key='LIAR test'):
    d = np.load(path, allow_pickle=True)
    return pd.Series(d[f'reply::{key}'], index=d[f'doc_id::{key}'].astype(str))

# 所有模型都按 5a 的 doc_id 顺序对齐 / everything is aligned on the 5a doc_ids
P = {'5a: no evidence': probs(LLM / 'llm_no_evidence.npz')}
ids = P['5a: no evidence'].index
P['5b: own evidence'] = probs(LLM / 'llm_evidence.npz').loc[ids]
P['control: another statement\'s evidence'] = probs(LLM / 'llm_random_evidence.npz').loc[ids]
P['TF-IDF (trained on LIAR)'] = probs(LIAR_RES / 'tfidf_liar.npz').loc[ids]
for s in (42, 43, 44):
    P[f'DistilBERT seed {s}'] = probs(LIAR_RES / f'distilbert_seed{s}.npz').loc[ids]
y = liar.loc[ids, 'label'].to_numpy()
label6 = liar.loc[ids, 'label6']
print(len(ids), 'test statements | real share', round(y.mean(), 3))

1283 test statements | real share 0.567


## 2. Prompts and cost / 提示词与费用

5a 和 5b 的任务完全一样；5b 多了证据，并被要求"证据不相关时依靠常识"，同时报告证据是否相关（`evidence_relevant`）。对照组和 5b 用的是 **同一个提示词、同样多的 token**，只有证据的内容不同。

In [2]:
meta = {name: json.loads((LLM / f'llm_{name}.json').read_text())
        for name in ('no_evidence', 'evidence', 'random_evidence')}
print('5a system prompt:\n', meta['no_evidence']['system_prompt'], '\n')
print('5b / control system prompt:\n', meta['evidence']['system_prompt'], '\n')

# 价格：Haiku 4.5 输入 $1、输出 $5 每百万 token，Batch API 五折
# Price: Haiku 4.5 $1 / $5 per MTok, halved by the Batch API
pd.DataFrame({name: {'requests': m['n_requests'],
                     'input tokens': m['usage']['input_tokens'],
                     'output tokens': m['usage']['output_tokens'],
                     'unparsed / failed': m['unparsed_or_failed']['LIAR test'],
                     'cost USD': round((m['usage']['input_tokens'] * 1 + m['usage']['output_tokens'] * 5) / 1e6 / 2, 3)}
              for name, m in meta.items()}).T

5a system prompt:
 You are a professional political fact-checker. You will be given a statement made by a US political figure. Estimate the probability that a professional fact-checker would rate the statement as true, mostly true, or half true (rather than barely true, false, or "pants on fire"). Use your general knowledge. Reply with only an integer from 0 to 100. 

5b / control system prompt:
 You are a professional political fact-checker. You will be given a statement made by a US political figure, followed by passages retrieved from Wikipedia. Estimate the probability that a professional fact-checker would rate the statement as true, mostly true, or half true (rather than barely true, false, or "pants on fire"). Use the passages when they are relevant; when they are not, rely on your general knowledge. Also say whether the passages were relevant to checking the statement. 



,requests,input tokens,output tokens,unparsed / failed,cost USD
no_evidence,1283.0,322214.0,11547.0,0.0,0.190
evidence,1283.0,818928.0,20528.0,0.0,0.461
random_evidence,1283.0,818928.0,20528.0,0.0,0.461


对照组的 token 数与 5b **完全相同**：每份证据恰好被另一条陈述用了一次，所以总长度不变。3,849 条请求没有一条无法解析。第 5 步的全部 API 费用约 1.4 美元（另有约 0.24 美元用于生成维基百科检索词）。

The control uses exactly the same tokens as 5b (each evidence set is reused once). No reply failed to parse.

## 3. The evidence / 证据是什么样的

检索流程（`src/evidence.py`）：LLM 为每条陈述写 1–2 个检索词 → 维基百科搜索 → 每个页面切段 → 按与陈述的 TF-IDF 相似度选 3 段。

**防泄漏过滤**：LIAR 的标签来自 PolitiFact，维基百科有时会直接引用核查结论。以下段落在选择前被删除：
- 提到事实核查机构的段落（PolitiFact、FactCheck.org、"Pinocchios" 等）；
- 复述这条陈述并带有真假判断词的段落；
- 标题就是"虚假说法汇编"的文章（例如 *False or misleading statements by …*）；
- 参考文献条目（ISBN、"Retrieved …"、作者-年份引用）。

In [3]:
ev = load_evidence('test')
removed = pd.Series({k: sum((v.get('removed') or {}).get(k, 0) for v in ev.values())
                     for k in ('factcheck', 'restated_verdict', 'verdict_article', 'reference')})
print('Paragraphs removed by each filter:\n', removed.to_string())
print('\nPassages per statement:', pd.Series([len(v['passages']) for v in ev.values()]).value_counts().to_dict())

Paragraphs removed by each filter:
 factcheck            641
restated_verdict      27
verdict_article     1405
reference           3624

Passages per statement: {3: 1281, 2: 1, 1: 1}


In [4]:
# 第一条测试陈述的证据，展示三类常见情况 / The first test statement's evidence shows three typical cases
example = ids[0]
print('Statement:', liar.loc[example, 'text'], '| label:', liar.loc[example, 'label6'])
print('Queries:', ev[example]['queries']['llm'])
for p in ev[example]['passages']:
    print(f"\n- ({p['title']}) {p['text'][:300]}...")

Statement: Building a wall on the U.S.-Mexico border will take literally years. | label: true
Queries: ['US Mexico border wall construction', 'border wall timeline project']

- (Mexico–United States border wall) About two-thirds of the U.S.–Mexico border runs along private or state-owned lands, and the federal government would need to acquire such land through purchase or seizure (eminent domain) to build any border wall. The "process is likely to cost the government millions and could take years of complex...

- (Russia–Ukraine border) In 2014, as the Ukrainian government lost Crimea and a portion of the Donbas to Russia and Russian-backed separatists, respectively, it unveiled a plan called "Project Wall" through which it sought to erect a fortified border barrier along the rest of the international border, with the goal of block...

- (Mexico–United States border wall) On December 18, 2021, the Texas Facilities Commission installed the first border wall panel on state-owned land for

这个例子里三段证据正好是三种情况：
- 第 1 段 **有用**：土地征收"可能需要数年的诉讼"，支持这句话（标签是 true）；
- 第 2 段 **无关**：检索词 "border wall" 匹配到了俄罗斯–乌克兰边境；
- 第 3 段 **时间不匹配**：讲的是 2021 年的事，而这句话是在 2016 年前说的。我们查的是今天的维基百科，它既缺少当年的细节，又包含说话时还没发生的事。

The example shows a useful passage, an unrelated page (Russia–Ukraine), and a time mismatch (2021 events for a pre-2016 statement; today's Wikipedia is used).

**泄漏审计**（在提交 5b 之前完成）：
- 不看标签，自动扫描全部 3,849 段，找"PolitiFact / 核查 / 虚假说法"等字样，发现两种漏网的泄漏方式（"虚假说法汇编"类文章 11 条、一个 Kessler 事实核查引用 1 条），据此加了上面的过滤规则，并重新检索。
- 人工阅读随机抽取的 30 段：**泄漏 0 段**；**真正包含判断所需事实的 3 段**，话题相关但没用的 12 段，完全无关的 15 段。

所以证据基本是干净的，但 **有用的证据很少，只有约 10%**。这一点决定了下面的结果。

Leak audit before submission: a label-blind scan found two leak channels (now filtered); a manual read of 30 random passages found 0 leaks, 3 useful, 12 on-topic but unhelpful, 15 irrelevant.

## 4. Results / 结果

In [5]:
rows = {'always "true" (majority)': {'accuracy %': round(100 * y.mean(), 1), 'balanced accuracy %': 50.0,
                                     'AUC': 0.5, 'predicted real %': 100.0}}
for name, p in P.items():
    m = evaluate(y, p.to_numpy())
    rows[name] = {k: m[k] for k in ('accuracy %', 'balanced accuracy %', 'AUC', 'predicted real %')}
    rows[name]['AUC 95% CI'] = bootstrap_ci(y, p.to_numpy())['AUC 95% CI']
results = pd.DataFrame(rows).T
results

,accuracy %,balanced accuracy %,AUC,predicted real %,AUC 95% CI
"always ""true"" (majority)",56.7,50.0,0.5,100.0,NaN
5a: no evidence,61.1,62.3,0.655,42.4,"(0.628, 0.682)"
5b: own evidence,53.5,57.5,0.647,21.0,"(0.618, 0.676)"
control: another statement's evidence,49.9,54.8,0.613,13.6,"(0.585, 0.643)"
TF-IDF (trained on LIAR),62.5,58.9,0.672,78.6,"(0.644, 0.7)"
DistilBERT seed 42,64.4,62.0,0.684,69.5,"(0.656, 0.712)"
DistilBERT seed 43,65.0,63.8,0.687,61.1,"(0.659, 0.716)"
DistilBERT seed 44,63.5,62.2,0.675,61.7,"(0.647, 0.703)"


**两点一眼可见**：
1. 三个 Haiku 实验的 AUC（0.613–0.655）都低于只看文字的 TF-IDF（0.672）和 DistilBERT（约 0.68）。下一节的配对检验显示，5a、5b 与它们的差距大多分不出高下，但 **没有一个超过它们**。
2. 5b 和对照组 **很少判"真"**（21% 和 14%，而真实比例是 57%），所以准确率低于"全部猜真"。

Haiku's AUCs (0.613–0.655) do not exceed the text-only models, and 5b / control rarely say "true" (21% / 14% vs 57% in the data), so their accuracy falls below the majority baseline.

## 5. Paired comparisons / 配对比较

同一批陈述上的 AUC 差（配对 bootstrap，1,000 次）。置信区间不包含 0 才算可靠的差距。

In [6]:
def diff(a, b):
    r = paired_bootstrap_auc_diff(y, P[a].to_numpy(), P[b].to_numpy())
    return {'A': a, 'B': b, 'AUC(A) - AUC(B)': r['AUC difference (A - B)'], '95% CI': r['95% CI'],
            'reliable?': 'yes' if (r['95% CI'][0] > 0 or r['95% CI'][1] < 0) else 'no'}

pairs = [('5b: own evidence', '5a: no evidence'),
         ('control: another statement\'s evidence', '5a: no evidence'),
         ('5b: own evidence', 'control: another statement\'s evidence'),
         ('5a: no evidence', 'TF-IDF (trained on LIAR)'),
         ('5b: own evidence', 'TF-IDF (trained on LIAR)')]
pairs += [('5a: no evidence', f'DistilBERT seed {s}') for s in (42, 43, 44)]
pairs += [('5b: own evidence', f'DistilBERT seed {s}') for s in (42, 43, 44)]
pd.DataFrame([diff(a, b) for a, b in pairs])

,A,B,AUC(A) - AUC(B),95% CI,reliable?
0,5b: own evidence,5a: no evidence,-0.008,"(-0.034, 0.018)",no
1,control: another statement's evidence,5a: no evidence,-0.042,"(-0.069, -0.014)",yes
2,5b: own evidence,control: another statement's evidence,0.034,"(0.008, 0.06)",yes
3,5a: no evidence,TF-IDF (trained on LIAR),-0.016,"(-0.055, 0.022)",no
4,5b: own evidence,TF-IDF (trained on LIAR),-0.025,"(-0.062, 0.013)",no
5,5a: no evidence,DistilBERT seed 42,-0.028,"(-0.066, 0.009)",no
6,5a: no evidence,DistilBERT seed 43,-0.031,"(-0.07, 0.007)",no
7,5a: no evidence,DistilBERT seed 44,-0.020,"(-0.059, 0.017)",no
8,5b: own evidence,DistilBERT seed 42,-0.037,"(-0.075, 0.002)",no
9,5b: own evidence,DistilBERT seed 43,-0.040,"(-0.077, -0.001)",yes


**对照组把 "5b ≈ 5a" 拆成了两个方向相反的作用**：

```
5a（不给证据）                        AUC 0.655
  ↓ 提示里放进证据，本身让判断变差          −0.042  （对照组 − 5a，可靠）
对照组（证据和陈述无关）                AUC 0.613
  ↓ 证据内容和陈述对得上，带来提升          +0.034  （5b − 对照组，可靠）
5b（自己的证据）                       AUC 0.647   → 两者几乎抵消，5b − 5a 分不出高下
```

如果没有对照组，只会看到"证据没用"，而这个结论是错的：**证据的内容有用，但"给证据"这个形式本身带来了代价**。

The control splits "5b ≈ 5a" into two opposite effects: putting passages in the prompt costs ~0.04 AUC (control vs 5a), while the matching content recovers ~0.03 (5b vs control).

## 6. Where the cost comes from: scepticism / 代价从哪里来：过度怀疑

如果模型把"证据里没提到"当成"说法是错的"，那么看到证据之后，它对 **所有** 真假等级的打分都会下降。

In [7]:
llm = ['5a: no evidence', '5b: own evidence', 'control: another statement\'s evidence']
by_level = pd.DataFrame({name: P[name].groupby(label6).mean() for name in llm}).reindex(ORDER).round(3)
by_level.insert(0, 'n', label6.value_counts().reindex(ORDER))
by_level.loc['all statements'] = [len(ids)] + [round(P[n].mean(), 3) for n in llm]
by_level

,n,5a: no evidence,5b: own evidence,control: another statement's evidence
label6,,,,
pants-fire,92.0,0.244,0.168,0.165
false,250.0,0.360,0.267,0.229
barely-true,214.0,0.409,0.314,0.264
half-true,267.0,0.448,0.331,0.253
mostly-true,249.0,0.539,0.398,0.326
true,211.0,0.546,0.428,0.370
all statements,1283.0,0.444,0.333,0.277


- 看到证据后，**每个等级**的平均概率都下降了约 0.08–0.14，对照组降得更多。这就是"过度怀疑"：维基百科很少记录 LIAR 里那么细的事实，于是"没找到支持"被当成了"不成立"。
- 但三个实验 **从 pants-fire 到 true 的顺序都基本保持**，说明模型的 *排序* 仍然有信息，问题主要出在 *整体偏移*。整体偏移会拉低准确率，却不影响 AUC，这也是 5b 的 AUC 接近 5a、准确率却低很多的原因。

Evidence lowers the mean probability at every truth level (more so with unrelated evidence): the model treats "not supported by the passages" as "false". The ordering across levels survives, so the damage is mostly a shift (which hurts accuracy) rather than a loss of ranking (AUC).

## 7. When the model said the evidence was relevant / 模型认为证据相关时

5b 和对照组都要求模型说明证据是否相关（`evidence_relevant`）。

In [8]:
rel = {name: replies(LLM / f'llm_{name}.npz').loc[ids].map(L.parse_relevant)
       for name in ('evidence', 'random_evidence')}
print('Share judged relevant: 5b', round(100 * rel['evidence'].mean(), 1), '% | control',
      round(100 * rel['random_evidence'].mean(), 1), '%')

Share judged relevant: 5b 38.4 % | control 0.1 %


对照组里几乎没有一条被判为"相关"（0.1%），说明模型确实能分辨证据是否对口，对照组的设计是有效的。

下面按 **5b 中模型的判断** 把陈述分成两组，比较三个实验在同一组陈述上的表现。注意这是事后分组（分组依据是 5b 的回答），只用来帮助理解，不是因果结论。

In [9]:
group = rel['evidence'].map({True: 'judged relevant (5b)', False: 'judged not relevant (5b)'})
rows = []
for g in group.dropna().unique():
    mask = (group == g).to_numpy()
    row = {'group': g, 'n': int(mask.sum()), 'real %': round(100 * y[mask].mean(), 1)}
    for name in llm:
        row[f'AUC {name.split(":")[0]}'] = round(roc_auc_score(y[mask], P[name].to_numpy()[mask]), 3)
    for a, b in (('5b: own evidence', '5a: no evidence'), ('5b: own evidence', 'control: another statement\'s evidence')):
        r = paired_bootstrap_auc_diff(y[mask], P[a].to_numpy()[mask], P[b].to_numpy()[mask])
        row[f'{a.split(":")[0]} - {b.split(":")[0]}'] = f"{r['AUC difference (A - B)']} {r['95% CI']}"
    rows.append(row)
pd.DataFrame(rows).set_index('group')

,n,real %,AUC 5a,AUC 5b,AUC control,5b - 5a,5b - control
group,,,,,,,
judged relevant (5b),493,59.8,0.661,0.658,0.634,"-0.003 (-0.046, 0.045)","0.024 (-0.026, 0.072)"
judged not relevant (5b),790,54.7,0.649,0.634,0.594,"-0.015 (-0.046, 0.015)","0.04 (0.01, 0.073)"


**即使在模型自己认为"证据相关"的 38% 陈述里，5b 也没有超过 5a。** 这和人工阅读的结论一致：维基百科的段落大多只是"话题相关"，很少包含决定真假的那个具体事实。

Even where the model judged the passages relevant, 5b does not beat 5a: the passages are mostly on-topic without containing the deciding fact.

## 8. Sensitivity: without half-true / 敏感性：去掉 half-true

half-true 被算作"真"，但它本身含糊。去掉这一级后再算一次 AUC，看结论是否改变。

In [10]:
keep = (label6 != 'half-true').to_numpy()
pd.DataFrame({name: {'AUC, all statements': round(roc_auc_score(y, P[name]), 3),
                     f'AUC without half-true (n={keep.sum()})': round(roc_auc_score(y[keep], P[name].to_numpy()[keep]), 3)}
              for name in P}).T

,"AUC, all statements",AUC without half-true (n=1016)
5a: no evidence,0.655,0.686
5b: own evidence,0.647,0.675
control: another statement's evidence,0.613,0.646
TF-IDF (trained on LIAR),0.672,0.697
DistilBERT seed 42,0.684,0.720
DistilBERT seed 43,0.687,0.723
DistilBERT seed 44,0.675,0.717


去掉 half-true 后所有模型的 AUC 都提高了约 0.02–0.04，但 **排序和结论都不变**：Haiku 没有超过只看文字的模型，5b 没有超过 5a，对照组最差。

Without half-true every AUC rises by ~0.02–0.04, but the ranking and conclusions are unchanged.

## 9. Step 5c: combining DistilBERT, TF-IDF and the LLM / 第 5c 步：把三个模型组合起来

Haiku 单独使用赢不了 DistilBERT，但两者的打分 **几乎不相关**：DistilBERT 学的是"说法的样子"，Haiku 用的是世界知识。两个水平接近、依据不同的模型，组合后通常比任何一个都好。

**做法**（`src/ensemble.py`）：
1. 在 LIAR **val** 上补跑了 Haiku 5a（1,284 条，约 0.19 美元）。
2. 把每个模型的概率转成 logit，用逻辑回归学习权重。组合器只在 val 上训练：DistilBERT 和 TF-IDF 已经见过训练集，在训练集上的打分过于自信。
3. **事先定好的选择规则**：每个候选在 val 上做 5 折交叉验证（打分的那一折组合器没见过），选 3 个 DistilBERT 种子平均 AUC 最高的候选。
4. 选择结果先保存（`results/llm/combination_val.json`），再在 test 上 **只评估这一个候选**。

Haiku alone does not beat DistilBERT, but their scores are nearly uncorrelated. The combiner (logistic regression on logits) is fitted on LIAR val only; the candidate is chosen by 5-fold CV AUC on val, saved, and only then evaluated once on test.

In [11]:
sel = json.loads((LLM / 'combination_val.json').read_text())
print('Selection rule:', sel['rule'])
print('Chosen on val:', sel['chosen'], '\n')
print('Spearman correlation of the models on val:')
print(pd.DataFrame(sel['spearman']).round(2).to_string(), '\n')
pd.DataFrame(sel['cv_auc']).T.sort_values('mean CV AUC', ascending=False)

Selection rule: highest mean 5-fold CV AUC on LIAR val over DistilBERT seeds
Chosen on val: DistilBERT + TF-IDF + Haiku 

Spearman correlation of the models on val:
                   tfidf  distilbert_seed42  distilbert_seed43  distilbert_seed44  haiku
tfidf               1.00               0.75               0.73               0.77   0.12
distilbert_seed42   0.75               1.00               0.92               0.89   0.15
distilbert_seed43   0.73               0.92               1.00               0.89   0.17
distilbert_seed44   0.77               0.89               0.89               1.00   0.13
haiku               0.12               0.15               0.17               0.13   1.00 



,seed 42,seed 43,seed 44,mean CV AUC
DistilBERT + TF-IDF + Haiku,0.7457,0.7401,0.7379,0.7412
DistilBERT + Haiku,0.7430,0.7332,0.7326,0.7363
TF-IDF + Haiku,0.7361,0.7361,0.7361,0.7361
DistilBERT + TF-IDF,0.6979,0.6928,0.6875,0.6927
DistilBERT,0.6920,0.6819,0.6764,0.6834
TF-IDF,0.6803,0.6803,0.6803,0.6803
Haiku,0.6658,0.6658,0.6658,0.6658


- 在 val 上，Haiku 和三个只看文字的模型的相关系数只有 0.12–0.17，而 TF-IDF 和 DistilBERT 之间是 0.73–0.77。
- 所以 **DistilBERT + TF-IDF**（两个"看文字"的模型）只比 DistilBERT 高约 0.01，而 **任何一个文字模型 + Haiku** 都高出约 0.05。
- 被选中的是 DistilBERT + TF-IDF + Haiku。值得注意的是，**TF-IDF + Haiku**（不需要 GPU 训练）几乎一样好。

In [12]:
test_rec = json.loads((LLM / 'combination_test.json').read_text())
assert test_rec['chosen_on_val'] == sel['chosen']          # test 只评估 val 选中的候选 / only the val choice
rows = {}
for seed, r in test_rec['results'].items():
    rows[seed] = {'weights (standardised logits)': r['weights (standardised logits)'],
                  'DistilBERT alone: AUC': r['DistilBERT alone']['AUC'],
                  'combined: AUC': r['combined']['AUC'],
                  'combined - DistilBERT': f"{r['combined - DistilBERT']['AUC difference (A - B)']} {tuple(r['combined - DistilBERT']['95% CI'])}",
                  'combined - TF-IDF': f"{r['combined - TF-IDF']['AUC difference (A - B)']} {tuple(r['combined - TF-IDF']['95% CI'])}",
                  'combined - Haiku 5a': f"{r['combined - Haiku']['AUC difference (A - B)']} {tuple(r['combined - Haiku']['95% CI'])}",
                  'DistilBERT alone: accuracy %': r['DistilBERT alone']['accuracy %'],
                  'combined: accuracy %': r['combined']['accuracy %'],
                  'combined: predicted real %': r['combined']['predicted real %']}
pd.DataFrame(rows).T

,weights (standardised logits),DistilBERT alone: AUC,combined: AUC,combined - DistilBERT,combined - TF-IDF,combined - Haiku 5a,DistilBERT alone: accuracy %,combined: accuracy %,combined: predicted real %
seed 42,"{'distilbert_seed42': 0.44, 'tfidf': 0.339, 'haiku': 0.628}",0.684,0.728,"0.044 (0.023, 0.065)","0.057 (0.033, 0.078)","0.073 (0.052, 0.093)",64.4,67.1,57.6
seed 43,"{'distilbert_seed43': 0.344, 'tfidf': 0.419, 'haiku': 0.622}",0.687,0.729,"0.042 (0.02, 0.064)","0.057 (0.035, 0.079)","0.073 (0.052, 0.094)",65.0,67.5,58.5
seed 44,"{'distilbert_seed44': 0.275, 'tfidf': 0.451, 'haiku': 0.638}",0.675,0.725,"0.05 (0.027, 0.073)","0.053 (0.031, 0.076)","0.07 (0.049, 0.089)",63.5,66.4,58.5


In [13]:
# 组合模型是否也随六个真假等级变化？/ Does the combination follow the six truth levels?
comb = pd.Series(test_rec['results']['seed 42']['prob'], index=test_rec['doc_ids']).loc[ids]
pd.DataFrame({'n': label6.value_counts().reindex(ORDER),
              'combined (seed 42): predicted real %': ((comb > 0.5).groupby(label6).mean() * 100).reindex(ORDER).round(1),
              'DistilBERT seed 42: predicted real %': ((P['DistilBERT seed 42'] > 0.5).groupby(label6).mean() * 100).reindex(ORDER).round(1),
              '5a: predicted real %': ((P['5a: no evidence'] > 0.5).groupby(label6).mean() * 100).reindex(ORDER).round(1)})

,n,combined (seed 42): predicted real %,DistilBERT seed 42: predicted real %,5a: predicted real %
label6,,,,
pants-fire,92,23.9,40.2,16.3
false,250,36.4,53.6,26.8
barely-true,214,48.6,65.4,35.5
half-true,267,59.2,71.9,41.2
mostly-true,249,78.7,85.5,59.4
true,211,79.1,83.4,60.7


## 10. Conclusions / 结论

| LIAR test（1,283 条，全部猜真 = 56.7%） | 准确率 | AUC |
|---|---:|---:|
| TF-IDF | 62.5% | 0.672 |
| DistilBERT（3 个种子） | 63.5–65.0% | 0.675–0.687 |
| Haiku 5a：不给证据 | 61.1% | 0.655 |
| Haiku 5b：维基百科证据 | 53.5% | 0.647 |
| Haiku 对照组：别的陈述的证据 | 49.9% | 0.613 |
| **5c：DistilBERT + TF-IDF + Haiku 5a（3 个种子）** | **66.4–67.5%** | **0.725–0.729** |

**1. 大模型单独使用，没有超过只看文字的模型。** Haiku 凭自身知识（5a）的 AUC 是 0.655，和 TF-IDF、DistilBERT 分不出高下。这也说明它没有简单地"背出" PolitiFact 的结论，否则分数会高得多。

**2. 维基百科证据没有带来总体提升，但对照组说明了原因。** 5b（0.647）和 5a 分不出高下。对照组把它拆成两个方向相反的作用：
- 证据的 **内容** 有用：对的证据比无关证据可靠地好 +0.034；
- 但 **"给证据"这个形式** 让模型过度怀疑（−0.042）：它把"资料里没提到"当成"说法是错的"，只把 21% 的陈述判为真。

再加上证据本身质量低（人工抽查 30 段，只有 3 段包含判断所需的事实），两者抵消，总体没有变化。

**3. 把大模型和文字模型组合起来，突破了 0.68 的上限（5c）。** Haiku 和文字模型的打分几乎不相关（val 上 0.12–0.17）：一个用世界知识，一个看说法的样子。组合器只在 val 上训练和选择，在 test 上：
- AUC 0.725–0.729，比 DistilBERT **可靠地高 0.042–0.050**（3 个种子的置信区间都不包含 0），也可靠地高于 TF-IDF（+0.053–0.057）和 Haiku（+0.070–0.073）；
- 准确率 66.4–67.5%，判为真的比例约 58%，接近真实的 56.7%；
- 对"彻底的谎言"（pants-fire）只接受 24%，DistilBERT 是 40%。

**所以：单独换成大模型不是答案，"语言模式 + 世界知识"的组合才是。** 这是整个项目第一次在没有来源捷径的数据上明显超过只看文字的模型。

**局限**
- 只用了 Haiku 4.5 一个大模型，以及今天的维基百科（缺少当年的细节，又包含说话时还没发生的事）。
- Haiku 的训练数据可能包含 PolitiFact 网页。5a 单独只有 0.655，说明它没有简单地背答案，但不能排除组合中有一部分提升来自"部分记忆"。
- 组合器在 val 上训练；DistilBERT 和 TF-IDF 的超参数也是在 val 上选的，它们在 val 上的分数略微偏高，可能分到了略大的权重。这只会让组合 *低估* LLM 的作用。
- val 中真陈述占 52.0%，test 中占 56.7%，阈值是在 val 上学到的，所以 test 上判为真的比例（约 58%）和真实比例接近，但不完全一致。
- 清洗步骤删掉了 16 条陈述（0.12%）中的网址和账号名；5a、5b 看到的是同样的文本，所以比较是公平的。
- half-true 归为"真"有争议；去掉它之后所有结论都不变。

**下一步**（同样只在 val 上决定）：修正"过度怀疑"的提示词、换更大的模型，检验它们能否让组合 **进一步** 提高；然后做错误分析（`09`）。

---

1. **An LLM alone does not beat text-only models**: Haiku without evidence reaches AUC 0.655, indistinguishable from TF-IDF and DistilBERT (and too low to suggest it simply recalls PolitiFact verdicts).
2. **Wikipedia evidence gives no net gain, and the control shows why**: the right passages beat unrelated ones (+0.034), but adding passages makes the model sceptical (−0.042; it calls only 21% of statements true), and only ~10% of passages contain the deciding fact.
3. **Combining the LLM with the text models breaks the 0.68 ceiling** (5c): the scores are nearly uncorrelated (0.12–0.17 on val). Chosen and fitted on val only, DistilBERT + TF-IDF + Haiku reaches test AUC 0.725–0.729 and 66.4–67.5% accuracy, reliably above DistilBERT for every seed (+0.042 to +0.050). Language patterns and world knowledge are complementary; a bigger model on its own is not the answer.